<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/DecoderSingleHead_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch
import torch
from torch import nn
import math
import torch.nn.functional as F

In [3]:
# class Decoder(nn.Module) :
#     def __init__(self,d_head,seq_len):
#       super().__init__()
#       self.d_head = d_head
#       self.seq_len = seq_len

#       #masked-multihead parameters
#       self.Wk = nn.Linear(d_head,d_head)
#       self.Wq = nn.Linear(d_head,d_head)
#       self.Wv = nn.Linear(d_head,d_head)
#       self.norm1 = nn.LayerNorm(d_head)

#       self.ffn = nn.Sequential(
#           nn.Linear(d_head,d_head*4),
#           nn.GELU(),
#           nn.Linear(d_head*4,d_head)
#       )

#       #multihead parameters
#       self.Wk_multi = nn.Linear(d_head,d_head)
#       self.Wq_multi = nn.Linear(d_head,d_head)
#       self.Wv_multi = nn.Linear(d_head,d_head)
#       self.norm2 = nn.LayerNorm(d_head)

#       #final
#       self.norm3 = nn.LayerNorm(d_head)
#       self.LinearLayer = nn.Linear(d_head,d_head)

#     def maskedSelfAttention(self,input_embedings_ids,padding_mask=None):

#       K = self.Wk(input_embedings_ids)
#       Q = self.Wq(input_embedings_ids)
#       V = self.Wv(input_embedings_ids)

#       attention_scores = Q @ K.transpose(-2,-1) #row become column and column become row

#       scaled_score = attention_scores / math.sqrt(K.shape[-1])

#       if padding_mask is not None:
#         scaled_score = scaled_score.masked_fill(
#              padding_mask == 0,
#             -1e9
#           )

#       look_ahead_matrix = torch.triu(torch.ones(self.seq_len,self.seq_len),dim=1)

#       look_ahead_mask = look_ahead_matrix.masked_fill(
#           look_ahead_matrix == 1,
#           -1e9
#       )

#       combined_mask = scaled_score + look_ahead_mask
#       attention_weight = F.softmax(combined_mask,dim =-1)
#       final_attention = attention_weight @ V

#       return final_attention


#     def crossMultiheadSelfAttention(self,encoder_output,normalized,padding_mask=None):

#       k_multi = self.Wk_multi(encoder_output)
#       q_multi = self.Wq_multi(normalized)
#       v_multi = self.Wv_multi(encoder_output)

#       attention_scores = q_multi @ k_multi.transpose(-2,-1)
#       scaled_score = attention_scores / math.sqrt(k_multi.shape[-1])

#       if padding_mask is not None:
#         scaled_score = scaled_score.masked_fill(
#             padding_mask == 0,
#             -1e9
#         )

#       attention_weight = F.softmax(scaled_score,dim =-1)
#       final_attention = attention_weight @ v_multi

#       return final_attention


#     def forward(self,input_embedings_ids,encoder_output,padding_mask=None):

#       #masked-multi-head-attention
#       attention_output = self.maskedSelfAttention(
#           input_embedings_ids,
#           padding_mask
#         )

#       #Add Residual connection
#       residual_1 = attention_output + input_embedings_ids

#       #normalize
#       masked_multihead_output = self.norm1(residual_1)

#       #multi-head attention
#       cross_multihead_attention = self.crossMultiheadSelfAttention(
#           encoder_output,
#           masked_multihead_output,
#           padding_mask
#         )

#       #Add Residual connection
#       residual_2 = cross_multihead_attention + masked_multihead_output

#       #normalize
#       normalized2 = self.norm2(residual_2)

#       #feed-forward-network
#       ffn = self.ffn(normalized2)

#       #Add Residual connection
#       residual_3 = ffn + normalized2

#       #final output
#       final_normalized_output = self.norm3(residual_3)

#       linear_output = self.LinearLayer(final_normalized_output)
#       return linear_output


### Self-Supervised Learning for Next-Word Prediction

**Self-supervised learning** in the context of language models means that the model learns from the structure of the data itself, without needing explicit human-labeled annotations. For **next-word prediction**, the task is to predict the next word in a sequence given the preceding words. The 'labels' for this task are automatically generated from the input text.

**How it works:**
1.  **Input Data:** You start with a large corpus of unannotated text (e.g., books, articles, web pages).
2.  **Sequence Generation:** For training, you take segments of this text. For each segment, you create input-target pairs:
    *   **Input Sequence:** A sequence of tokens (words or subwords) up to a certain point.
    *   **Target Sequence:** The very next token(s) that immediately follow the input sequence.
    *   For example, if your text is "The quick brown fox jumps."
        *   Input: "The" -> Target: "quick"
        *   Input: "The quick" -> Target: "brown"
        *   Input: "The quick brown" -> Target: "fox"
        *   Input: "The quick brown fox" -> Target: "jumps"
3.  **Masked Attention (in your Decoder):** Your `maskedSelfAttention` layer uses a 'look-ahead mask' to ensure that during training, the model can only attend to tokens that appear *before* the current token it's trying to predict. This prevents information leakage from future tokens.
4.  **Cross-Attention (in your Decoder):** Your `crossMultiheadSelfAttention` layer expects an `encoder_output`. In a typical encoder-decoder Transformer, this would come from an encoder processing a different input sequence. For a purely self-supervised *decoder-only* language model (like GPT), this cross-attention layer is often removed, or it attends to the *same* input sequence as the masked self-attention (though configured differently), or to a fixed context. For demonstration purposes with your current `Decoder` structure, we'll pass the input embeddings themselves to simulate a context.
5.  **Prediction Head:** The output of your `Decoder` (which has `d_head` dimensions) needs to be projected onto the `vocab_size` (number of unique words/tokens in your vocabulary). This is typically done with a final linear layer.
6.  **Loss Function:** A standard `CrossEntropyLoss` is used to compare the model's predicted probabilities for the next word against the actual next word. The model adjusts its weights to minimize this loss, thereby learning to better predict subsequent tokens.

In [5]:
class LanguageModel(nn.Module):
  def __init__(self, vocab_size, d_head, seq_len):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_head) # Token embeddings
    self.decoder_block = Decoder(d_head, seq_len) # User's Decoder block
    self.output_linear = nn.Linear(d_head, vocab_size) # Project decoder output to vocab size

  def forward(self, input_token_ids, padding_mask=None):
    # 1. Embed input tokens
    input_embeds = self.embedding(input_token_ids)

    # 2. Pass through the decoder block
    # For next-word prediction in a self-supervised context with this decoder structure,
    # we can pass the input_embeds also as the 'encoder_output' for the cross-attention.
    # This allows the decoder to run as provided, although a true decoder-only LM might
    # remove or modify the cross-attention layer.
    decoder_output = self.decoder_block(
        input_embeds,
        input_embeds, # Using input_embeds as encoder_output for demonstration
        padding_mask
    )

    # 3. Project decoder output to vocabulary size for next-word prediction
    logits = self.output_linear(decoder_output)
    return logits

In [8]:
import torch
from torch import nn
import math
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter

# Corrected Decoder class definition (remains unchanged from last turn)
class Decoder(nn.Module) :
    def __init__(self,d_head,seq_len):
      super().__init__()
      self.d_head = d_head
      self.seq_len = seq_len

      #masked-multihead parameters
      self.Wk = nn.Linear(d_head,d_head)
      self.Wq = nn.Linear(d_head,d_head)
      self.Wv = nn.Linear(d_head,d_head)
      self.norm1 = nn.LayerNorm(d_head)

      self.ffn = nn.Sequential(
          nn.Linear(d_head,d_head*4),
          nn.GELU(),
          nn.Linear(d_head*4,d_head)
      )

      #multihead parameters
      self.Wk_multi = nn.Linear(d_head,d_head)
      self.Wq_multi = nn.Linear(d_head,d_head)
      self.Wv_multi = nn.Linear(d_head,d_head)
      self.norm2 = nn.LayerNorm(d_head)

      #final
      self.norm3 = nn.LayerNorm(d_head)
      self.LinearLayer = nn.Linear(d_head,d_head)

    def maskedSelfAttention(self,input_embedings_ids,padding_mask=None):

      K = self.Wk(input_embedings_ids)
      Q = self.Wq(input_embedings_ids)
      V = self.Wv(input_embedings_ids)

      attention_scores = Q @ K.transpose(-2,-1) #row become column and column become row

      scaled_score = attention_scores / math.sqrt(K.shape[-1])

      if padding_mask is not None:
        scaled_score = scaled_score.masked_fill(
             padding_mask == 0,
            -1e9
          )

      # FIX: Changed 'dim=1' to 'diagonal=1'
      look_ahead_matrix = torch.triu(torch.ones(self.seq_len,self.seq_len),diagonal=1)

      look_ahead_mask = look_ahead_matrix.masked_fill(
          look_ahead_matrix == 1,
          -1e9
      )

      combined_mask = scaled_score + look_ahead_mask
      attention_weight = F.softmax(combined_mask,dim =-1)
      final_attention = attention_weight @ V

      return final_attention


    def crossMultiheadSelfAttention(self,encoder_output,normalized,padding_mask=None):

      k_multi = self.Wk_multi(encoder_output)
      q_multi = self.Wq_multi(normalized)
      v_multi = self.Wv_multi(encoder_output)

      attention_scores = q_multi @ k_multi.transpose(-2,-1)
      scaled_score = attention_scores / math.sqrt(k_multi.shape[-1])

      if padding_mask is not None:
        scaled_score = scaled_score.masked_fill(
            padding_mask == 0,
            -1e9
        )

      attention_weight = F.softmax(scaled_score,dim =-1)
      final_attention = attention_weight @ v_multi

      return final_attention


    def forward(self,input_embedings_ids,encoder_output,padding_mask=None):

      #masked-multi-head-attention
      attention_output = self.maskedSelfAttention(
          input_embedings_ids,
          padding_mask
        )

      #Add Residual connection
      residual_1 = attention_output + input_embedings_ids

      #normalize
      masked_multihead_output = self.norm1(residual_1)

      #multi-head attention
      cross_multihead_attention = self.crossMultiheadSelfAttention(
          encoder_output,
          masked_multihead_output,
          padding_mask
        )

      #Add Residual connection
      residual_2 = cross_multihead_attention + masked_multihead_output

      #normalize
      normalized2 = self.norm2(residual_2)

      #feed-forward-network
      ffn = self.ffn(normalized2)

      #Add Residual connection
      residual_3 = ffn + normalized2

      #final output
      final_normalized_output = self.norm3(residual_3)

      linear_output = self.LinearLayer(final_normalized_output)
      return linear_output


# LanguageModel wrapper class (remains unchanged)
class LanguageModel(nn.Module):
  def __init__(self, vocab_size, d_head, seq_len):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_head) # Token embeddings
    self.decoder_block = Decoder(d_head, seq_len) # User's Decoder block
    self.output_linear = nn.Linear(d_head, vocab_size) # Project decoder output to vocab size

  def forward(self, input_token_ids, padding_mask=None):
    # 1. Embed input tokens
    input_embeds = self.embedding(input_token_ids)

    # 2. Pass through the decoder block
    decoder_output = self.decoder_block(
        input_embeds,
        input_embeds, # Using input_embeds as encoder_output for demonstration
        padding_mask
    )

    # 3. Project decoder output to vocabulary size for next-word prediction
    logits = self.output_linear(decoder_output)
    return logits


# --- Real English Paragraph and Preprocessing ---

# Sample English Paragraph
english_paragraph = (
    "Natural language processing (NLP) is a subfield of artificial intelligence "
    "(AI) that focuses on enabling computers to understand, interpret, and generate "
    "human language. It combines computational linguistics—rule-based modeling "
    "of human language—with machine learning, deep learning, and artificial intelligence. "
    "NLP has evolved significantly, from symbolic methods in the 1950s to statistical "
    "approaches in the 1980s and 1990s, and more recently, to neural networks and deep learning. "
    "Modern NLP applications include machine translation, spam filtering, sentiment analysis, "
    "and virtual assistants. The goal is to bridge the gap between human communication "
    "and computer understanding, making interactions more natural and intuitive."
)

# Simple Tokenization and Vocabulary Creation
def tokenize_and_build_vocab(text):
    # Convert to lowercase and split by non-alphanumeric characters
    tokens = [token.lower() for token in re.findall(r'\b\w+\b', text) if token.isalnum()]

    # Add special tokens
    special_tokens = ['<pad>', '<unk>', '<sos>', '<eos>']

    # Build vocabulary
    word_counts = Counter(tokens)
    # Ensure special tokens are at the beginning with specific IDs
    vocab_list = special_tokens + [word for word, _ in word_counts.most_common() if word not in special_tokens]

    word_to_idx = {word: i for i, word in enumerate(vocab_list)}
    idx_to_word = {i: word for i, word in enumerate(vocab_list)}

    return word_to_idx, idx_to_word, len(vocab_list), tokens

import re # Import regex module

word_to_idx, idx_to_word, vocab_size, all_tokens = tokenize_and_build_vocab(english_paragraph)

# Convert tokens to numerical sequences
def numericalize_text(tokens, word_to_idx, seq_len):
    numerical_sequence = [word_to_idx.get(token, word_to_idx['<unk>']) for token in tokens]

    # Split into chunks of seq_len for training
    sequences = []
    for i in range(0, len(numerical_sequence) - 1, seq_len):
        input_seq = numerical_sequence[i : i + seq_len]

        if len(input_seq) < seq_len:
            # Pad shorter sequences with <pad> token
            input_seq = input_seq + [word_to_idx['<pad>']] * (seq_len - len(input_seq))

        # Ensure it's exactly seq_len
        input_seq = input_seq[:seq_len]
        sequences.append(input_seq)

    # For targets, we need to shift the input sequences by one
    input_data = []
    target_data = []
    for seq in sequences:
        # Input is current sequence
        input_data.append(seq)
        # Target is next token for each position, shifted by one, with padding at the end
        target_data.append(seq[1:] + [word_to_idx['<pad>']])

    return torch.tensor(input_data, dtype=torch.long), torch.tensor(target_data, dtype=torch.long)

# Define hyperparameters for our training
d_head = 64        # Embedding dimension (matches d_head in your Decoder)
seq_len = 20       # Adjusted sequence length to be reasonable for a paragraph
batch_size = 2     # Number of sequences per batch
num_epochs = 100   # Increased epochs for a small dataset

# Prepare actual data
input_sequences_tensor, target_sequences_tensor = numericalize_text(all_tokens, word_to_idx, seq_len)

# Create a DataLoader (if you had more data)
# For this small example, we'll just use the tensors directly

print(f"Vocabulary size: {vocab_size}")
print(f"Number of training sequences: {input_sequences_tensor.shape[0]}")

# Instantiate the Decoder block
decoder_block_instance = Decoder(d_head=d_head, seq_len=seq_len)

# Instantiate the LanguageModel wrapper
model = LanguageModel(vocab_size, d_head, seq_len)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=word_to_idx['<pad>']) # Ignore padding token in loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting training loop with real paragraph data...")

# Training loop
model.train() # Set model to training mode
for epoch in range(num_epochs):
    total_loss = 0
    num_batches = 0

    # Iterate through data in batches
    for i in range(0, input_sequences_tensor.shape[0], batch_size):
        current_batch_inputs = input_sequences_tensor[i:i+batch_size]
        current_batch_targets = target_sequences_tensor[i:i+batch_size]

        if current_batch_inputs.shape[0] == 0: # Handle cases where batch might be empty
            continue

        optimizer.zero_grad()

        # Padding mask: create a mask based on the <pad> token
        # 1 where it's not padding, 0 where it is padding. The Decoder expects 0 for masked positions.
        # The Decoder's masked_fill uses -1e9 when input_embedings == 0, so here we need to ensure the padding_mask logic aligns.
        # Let's adjust the padding_mask to be 1 for valid tokens and 0 for padding (to be consistent with the original `padding_mask == 0` logic in Decoder)
        batch_padding_mask_bool = (current_batch_inputs != word_to_idx['<pad>']).unsqueeze(1).expand(-1, seq_len, -1) # Shape: (batch_size, seq_len, seq_len)

        # Forward pass
        logits = model(current_batch_inputs, batch_padding_mask_bool)

        # Reshape logits and targets for CrossEntropyLoss
        # logits shape: (batch_size, seq_len, vocab_size)
        # targets shape: (batch_size, seq_len)
        loss = criterion(logits.view(-1, vocab_size), current_batch_targets.view(-1))

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    if num_batches > 0:
        avg_loss = total_loss / num_batches
        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
    else:
        print(f"Epoch {epoch+1}/{num_epochs}, No batches processed.")

print("Training complete!")

# Example of making a prediction (inference)
print("\n--- Example Inference ---")
model.eval() # Set model to evaluation mode
with torch.no_grad():
    # Let's take a short phrase from our paragraph
    test_phrase = "natural language processing is"
    test_tokens = [token.lower() for token in re.findall(r'\b\w+\b', test_phrase) if token.isalnum()]

    # Convert to IDs and pad/truncate to seq_len
    input_ids = [word_to_idx.get(token, word_to_idx['<unk>']) for token in test_tokens]
    input_ids = input_ids[:seq_len] # Truncate if too long
    input_ids_padded = input_ids + [word_to_idx['<pad>']] * (seq_len - len(input_ids)) # Pad

    input_tensor = torch.tensor([input_ids_padded], dtype=torch.long)

    # Create padding mask for inference (based on actual length of the input phrase)
    inference_padding_mask_bool = (input_tensor != word_to_idx['<pad>']).unsqueeze(1).expand(-1, seq_len, -1)

    predicted_logits = model(input_tensor, inference_padding_mask_bool)
    predicted_probs = F.softmax(predicted_logits[0, -1, :], dim=-1) # Get prediction for the *last* valid token
    predicted_next_token_id = torch.argmax(predicted_probs).item()

    predicted_word = idx_to_word.get(predicted_next_token_id, '<unk>')

    print(f"Input phrase: '{test_phrase}'")
    print(f"Predicted next word: '{predicted_word}'")
    print("Note: With a small paragraph and simple model, predictions will be basic.")



Vocabulary size: 74
Number of training sequences: 5
Starting training loop with real paragraph data...
Epoch 1/100, Average Loss: 4.3041
Epoch 2/100, Average Loss: 4.0891
Epoch 3/100, Average Loss: 3.9045
Epoch 4/100, Average Loss: 3.7233
Epoch 5/100, Average Loss: 3.5416
Epoch 6/100, Average Loss: 3.3571
Epoch 7/100, Average Loss: 3.1686
Epoch 8/100, Average Loss: 2.9754
Epoch 9/100, Average Loss: 2.7778
Epoch 10/100, Average Loss: 2.5769
Epoch 11/100, Average Loss: 2.3751
Epoch 12/100, Average Loss: 2.1753
Epoch 13/100, Average Loss: 1.9810
Epoch 14/100, Average Loss: 1.7948
Epoch 15/100, Average Loss: 1.6182
Epoch 16/100, Average Loss: 1.4527
Epoch 17/100, Average Loss: 1.2993
Epoch 18/100, Average Loss: 1.1584
Epoch 19/100, Average Loss: 1.0294
Epoch 20/100, Average Loss: 0.9118
Epoch 21/100, Average Loss: 0.8053
Epoch 22/100, Average Loss: 0.7095
Epoch 23/100, Average Loss: 0.6237
Epoch 24/100, Average Loss: 0.5472
Epoch 25/100, Average Loss: 0.4794
Epoch 26/100, Average Loss: 0.4